Data Collection

In [1]:
import pandas as pd
import numpy as np
from tabulate import tabulate

In [2]:
df = pd.read_csv('DataBreaches(2004-2021).csv')

In [3]:
columns = ['Entity', 'Records', 'Organization type', 'Method']
subset_df = df[columns].copy()

#print the first 5 rows of subset dataframe to ensure data loaded correctly
print(subset_df.head(10)) 

                           Entity    Records   Organization type  \
0           21st Century Oncology    2200000          healthcare   
1                           500px   14870304   social networking   
2           Accendo Insurance Co.     175350          healthcare   
3      Adobe Systems Incorporated  152000000                tech   
4                      Adobe Inc.    7500000                tech   
5          Advocate Medical Group    4000000          healthcare   
6  AerServ (subsidiary of InMobi)      75000         advertising   
7      Affinity Health Plan, Inc.     344579          healthcare   
8                          Airtel  320000000  telecommunications   
9                      Air Canada      20000           transport   

                Method  
0               hacked  
1               hacked  
2        poor security  
3               hacked  
4        poor security  
5  lost / stolen media  
6               hacked  
7  lost / stolen media  
8        poor security  
9

Transform Records into categories

In [4]:
size = [0, 1000000, 50000000, np.inf]

record_labels = ['Small Breach', 'Medium Breach', 'Large Breach']
df['Data Breach'] = pd.cut(df['Records'], bins=size, labels=record_labels)

print(df[['Entity', 'Data Breach', 'Organization type', 'Method']].head(10))

                           Entity    Data Breach   Organization type  \
0           21st Century Oncology  Medium Breach          healthcare   
1                           500px  Medium Breach   social networking   
2           Accendo Insurance Co.   Small Breach          healthcare   
3      Adobe Systems Incorporated   Large Breach                tech   
4                      Adobe Inc.  Medium Breach                tech   
5          Advocate Medical Group  Medium Breach          healthcare   
6  AerServ (subsidiary of InMobi)   Small Breach         advertising   
7      Affinity Health Plan, Inc.   Small Breach          healthcare   
8                          Airtel   Large Breach  telecommunications   
9                      Air Canada   Small Breach           transport   

                Method  
0               hacked  
1               hacked  
2        poor security  
3               hacked  
4        poor security  
5  lost / stolen media  
6               hacked  
7  lost

In [5]:
columns= ['Entity', 'Organization type', 'Data Breach', 'Method']
#Use dropna() to drop rows with missing text
clean_df = df.dropna(subset=['Organization type', 'Data Breach', 'Method']).copy()

clean_df['Profile_Set'] = clean_df.apply(lambda row: {str(row['Organization type']), str(row['Data Breach']), str(row['Method'])}, axis=1)

Display three main query entities

In [8]:
query_entities = ['Capital One', 'CareFirst BlueCross Blue Shield - Maryland', 'Facebook']
target_df = clean_df[clean_df['Entity'].isin(query_entities)]

columns_to_show = ['Entity', 'Organization type', 'Data Breach', 'Method']
target_profiles_clean = target_df[columns_to_show]
print(tabulate(target_profiles_clean, headers='keys', tablefmt='grid', showindex=False))

+--------------------------------------------+---------------------+---------------+------------------------+
| Entity                                     | Organization type   | Data Breach   | Method                 |
+============================================+=====================+===============+========================+
| Capital One                                | financial           | Large Breach  | unsecured S3 bucket    |
+--------------------------------------------+---------------------+---------------+------------------------+
| CareFirst BlueCross Blue Shield - Maryland | healthcare          | Medium Breach | hacked                 |
+--------------------------------------------+---------------------+---------------+------------------------+
| Facebook                                   | social network      | Medium Breach | accidentally published |
+--------------------------------------------+---------------------+---------------+------------------------+
| Facebook

Similarity Measurements

In [10]:
#Jaccard Similarity function
def jaccard_similarity(set1, set2):
    if not set1 or not set2:
        return 0  
    return len(set1 & set2) / len(set1 | set2)

In [11]:
def get_top_10_similar(entity_name, dataframe):
    target_row = dataframe[dataframe['Entity'] == entity_name].iloc[0]
    target_set = target_row['Profile_Set']
    
    dataframe['Similarity_Score'] = dataframe['Profile_Set'].apply(lambda x: jaccard_similarity(target_set, x))
    
    top_10 = dataframe[dataframe['Entity'] != entity_name].sort_values(by='Similarity_Score', ascending=False).head(10)
    
    return top_10[['Entity', 'Organization type', 'Method', 'Data Breach', 'Similarity_Score']]

Top 10 Data Breaches similar to Capital One

In [12]:
print("Top 10 Data Breaches Similar to Capital One:")
capital_one = get_top_10_similar('Capital One', clean_df)

print(tabulate(capital_one, headers='keys', tablefmt='grid', showindex=False))

Top 10 Data Breaches Similar to Capital One:
+--------------------------------------------------------------------+---------------------+------------------------+---------------+--------------------+
| Entity                                                             | Organization type   | Method                 | Data Breach   |   Similarity_Score |
+====================================================================+=====================+========================+===============+====================+
| Massive American business hack including 7-Eleven and Nasdaq       | financial           | hacked                 | Large Breach  |                0.5 |
+--------------------------------------------------------------------+---------------------+------------------------+---------------+--------------------+
| JP Morgan Chase                                                    | financial           | hacked                 | Large Breach  |                0.5 |
+------------------------

Top 10 Data Breaches Similar to Facebook

In [13]:
print("Top 10 Data Breaches Similar to Facebook:")
facebook = get_top_10_similar('Facebook', clean_df)

print(tabulate(facebook, headers='keys', tablefmt='grid', showindex=False))

Top 10 Data Breaches Similar to Facebook:
+----------------------------------------------+---------------------+------------------------+---------------+--------------------+
| Entity                                       | Organization type   | Method                 | Data Breach   |   Similarity_Score |
+==============================================+=====================+========================+===============+====================+
| Norwegian Tax Administration                 | government          | accidentally published | Medium Breach |                0.5 |
+----------------------------------------------+---------------------+------------------------+---------------+--------------------+
| Office of the Texas Attorney General         | government          | accidentally published | Medium Breach |                0.5 |
+----------------------------------------------+---------------------+------------------------+---------------+--------------------+
| Ministry of Education (Ch

Top 10 Data Breaches Similar to CareFirst BlueCross Blue Shield - Maryland

In [14]:
print("Top 10 Data Breaches Similar to CareFirst BlueCross Blue Shield - Maryland:")
careFirst = get_top_10_similar('CareFirst BlueCross Blue Shield - Maryland', clean_df)

print(tabulate(careFirst, headers='keys', tablefmt='grid', showindex=False))

Top 10 Data Breaches Similar to CareFirst BlueCross Blue Shield - Maryland:
+------------------------------------------+----------------------+----------+---------------+--------------------+
| Entity                                   | Organization type    | Method   | Data Breach   |   Similarity_Score |
+==========================================+======================+==========+===============+====================+
| UCLA Medical Center, Santa Monica        | healthcare           | hacked   | Medium Breach |                1   |
+------------------------------------------+----------------------+----------+---------------+--------------------+
| LifeLabs                                 | healthcare           | hacked   | Medium Breach |                1   |
+------------------------------------------+----------------------+----------+---------------+--------------------+
| Medical Informatics Engineering          | healthcare           | hacked   | Medium Breach |                1 